In [44]:
import cv2
import numpy as np
import time

In [45]:
confThreshold = 0.5  #Confidence threshold：信任閥值
nmsThreshold = 0.4   #Non-maximum suppression threshold，非極大刪減閥值：https://zhuanlan.zhihu.com/p/37489043
inpWidth = 640       #Width of network's input image，32的倍數，小則快（例如320），大則準（例如640）
inpHeight = 480      #Height of network's input image，32的倍數，小則快，大則準


In [46]:
# 檢測目標數字代表webcam(例如0,1....)，網址代表線上影像，台灣即時影像列表：https://tw.live/
#target="https://tw.live/cam/?id=BOT349"
#target= 0 # "https://thbcctv06.thb.gov.tw/T9-38K+300" #target= "https://thbcctv07.thb.gov.tw/T14A-32K+781"
#target="https://cctv.bote.gov.taipei:8501/mjpeg/133"
target = "image/face.jpg"
cap = cv2.VideoCapture(target)

In [47]:
# Give the configuration and weight files for the model and load the network using them.：載入Yolo神經網路架構
modelConfiguration = "YOLOWeight/yolov3-tiny.cfg"
modelWeights = "YOLOWeight/yolov3-tiny.weights"

In [48]:
# Load names of classes：coconames物件類別名稱
classesFile = "YOLOWeight/coco.names" 
#classesFile = "YOLOWeight-1/obj.names"
classes = None
with open(classesFile, 'rt') as f:
    classes = f.read().rstrip('\n').split('\n')

net = cv2.dnn.readNetFromDarknet(modelConfiguration, modelWeights)
net.setPreferableBackend(cv2.dnn.DNN_BACKEND_OPENCV)
net.setPreferableTarget(cv2.dnn.DNN_TARGET_CPU)


In [49]:
# 取得輸出網路
def getOutputsNames(net):
    layersNames = net.getLayerNames()
    return [layersNames[i-1] for i in net.getUnconnectedOutLayers()]

# 在圖上標示
def drawPred(classId, conf, left, top, right, bottom, frame):
   # classId：物件編號，conf：機率，left, top, right, bottom：物件位置框
   # Draw a bounding box.:畫方形
    cv2.rectangle(frame, (left, top), (right, bottom), (255, 178, 50), 3)
    label = '%.2f' % conf    
    # Get the label for the class name and its confidence
    if classes:
        assert(classId < len(classes))
        label = '%s:%s' % (classes[classId], label)

    #Display the label at the top of the bounding box
    labelSize, baseLine = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.5, 1)
    top = max(top, labelSize[1])
    cv2.rectangle(frame, (left, top - round(1.5*labelSize[1])), (left + round(1.5*labelSize[0]), top + baseLine), (255, 255, 255), cv2.FILLED)
    cv2.putText(frame, label, (left, top), cv2.FONT_HERSHEY_SIMPLEX, 0.75, (0,0,0), 1)


In [50]:
# 後處理：包括移除重複物件、刪除信任度低的物件，然後交給drawPred標示
def postprocess(frame, outs):
    frameHeight = frame.shape[0]
    frameWidth = frame.shape[1]
    classIds = []
    confidences = []
    boxes = []
    # 移除重複物件，保留最高信任度
    classIds = []
    confidences = []
    boxes = []
    for out in outs:
        for detection in out:
            scores = detection[5:]
            classId = np.argmax(scores)
            confidence = scores[classId]
            if confidence > confThreshold:
                center_x = int(detection[0] * frameWidth)
                center_y = int(detection[1] * frameHeight)
                width = int(detection[2] * frameWidth)
                height = int(detection[3] * frameHeight)
                left = int(center_x - width / 2)
                top = int(center_y - height / 2)
                classIds.append(classId)
                confidences.append(float(confidence))
                boxes.append([left, top, width, height])
   
    # 交給drawPred標示
    indices = cv2.dnn.NMSBoxes(boxes, confidences, confThreshold, nmsThreshold)
    for i in indices:
        i = i
        box = boxes[i]
        left = box[0]
        top = box[1]
        width = box[2]
        height = box[3]
        drawPred(classIds[i], confidences[i], left, top, left + width, top + height, frame)
        if classIds[i]==67:
            pass #cv2.imwrite("test.jpg", frame)
       



In [51]:
# Process inputs
while True:
    try:
        stime=time.time()
        hasFrame, frame = cap.read()
        blob = cv2.dnn.blobFromImage(frame, 1/255, (inpWidth, inpHeight), [0,0,0], 1, crop=False)
        #將影像輸入網路
        net.setInput(blob)
        #執行偵測，偵測物件=outs
        outs = net.forward(getOutputsNames(net))

        #後續處理
        postprocess(frame, outs)
        #計算fps
        etime = time.time()
        label = "fps=" + str(round(1/(etime-stime),2))
        cv2.putText(frame, label, (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255),2)
        frame=cv2.resize(frame,(640,480))
        cv2.imshow("YOLO", frame)
    except Exception as e:
        print(e) #印出錯誤訊息
        time.sleep(0.5)
        cap = cv2.VideoCapture(target)
    key=cv2.waitKey(1)#0=強制等待、1:等候1ms就跳過  更新影像
    # 按q離開
    if key & 0xFF == ord('q'):
        break
# 釋放攝影機
cap.release()

# 關閉所有 Opencv2 視窗
cv2.destroyAllWindows()


OpenCV(4.12.0) D:\a\opencv-python\opencv-python\opencv\modules\imgproc\src\resize.cpp:4208: error: (-215:Assertion failed) !ssize.empty() in function 'cv::resize'

OpenCV(4.12.0) D:\a\opencv-python\opencv-python\opencv\modules\imgproc\src\resize.cpp:4208: error: (-215:Assertion failed) !ssize.empty() in function 'cv::resize'

OpenCV(4.12.0) D:\a\opencv-python\opencv-python\opencv\modules\imgproc\src\resize.cpp:4208: error: (-215:Assertion failed) !ssize.empty() in function 'cv::resize'

OpenCV(4.12.0) D:\a\opencv-python\opencv-python\opencv\modules\imgproc\src\resize.cpp:4208: error: (-215:Assertion failed) !ssize.empty() in function 'cv::resize'

OpenCV(4.12.0) D:\a\opencv-python\opencv-python\opencv\modules\imgproc\src\resize.cpp:4208: error: (-215:Assertion failed) !ssize.empty() in function 'cv::resize'

OpenCV(4.12.0) D:\a\opencv-python\opencv-python\opencv\modules\imgproc\src\resize.cpp:4208: error: (-215:Assertion failed) !ssize.empty() in function 'cv::resize'

OpenCV(4.12.0) D